# O "programa dos sonhos" da FidelidadeScript

Antes de definir o analisador ou as demais fases do compilador, descrevemos o programa que queremos permitir que uma pessoa escreva. O exemplo combina uma compra, o cálculo do cashback, a concessão de pontos e operações comuns de uma conta de fidelidade.

Este programa é o contrato inicial da linguagem: as próximas etapas deverão torná-lo executável e validar as regras de negócio do domínio.

A primeira célula executável valida a ficha antes de definir o programa. Se os requisitos mínimos não forem atendidos, ela lança um erro.

In [ ]:
# ============================================================
# FICHA DE QUALIDADE — validar antes de definir o programa DSL
# ============================================================

import re

MINHA_FICHA = {
    "nome": "FidelidadeScript",
    "dominio": "programas de pontos e cashback",
    "tipos": ["inteiro", "decimal", "texto", "logico"],
    "comandos_dominio": [
        'creditar "<cliente>", <pontos>;',
        'resgatar "<cupom>", <pontos>;',
        'expirar "<cliente>", <pontos>;',
        'extrato "<cliente>";',
    ],
    "regras_de_negocio": [
        "percentual de cashback acima de 100% gera erro",
        "pontos negativos geram erro",
        "resgate acima do saldo disponível gera erro de execução",
    ],
    "programas_exemplo": [
        r'''
decimal compra = 250.0;
decimal percentual_cashback = 10.0;
decimal cashback = compra * percentual_cashback / 100.0;
inteiro pontos = 100;
se compra >= 200.0 {
    pontos = pontos + 50;
}
creditar "cliente_42", pontos;
extrato "cliente_42";
''',
        r'''
decimal percentual_cashback = 120.0;
inteiro pontos = -5;
se percentual_cashback > 100.0 {
    creditar "cliente_42", pontos;
}
''',
        r'''
inteiro pontos = 50;
creditar "cliente_42", pontos;
se pontos < 100 {
    resgatar "cupom_10", 100;
}
''',
    ],
}

def conferir_ficha(ficha):
    if not isinstance(ficha, dict):
        raise TypeError("A ficha da linguagem deve ser um dicionário.")

    def tem_texto(valor):
        return isinstance(valor, str) and bool(valor.strip())

    def quantidade_itens(campo):
        itens = ficha.get(campo)
        if not isinstance(itens, list):
            return 0
        return sum(tem_texto(item) for item in itens)

    nome = ficha.get("nome")
    dominio = ficha.get("dominio")
    identidade_valida = (
        tem_texto(nome)
        and tem_texto(dominio)
        and "todo" not in (nome + dominio).lower()
    )
    programas = ficha.get("programas_exemplo")
    tres_programas_validos = (
        isinstance(programas, list)
        and len(programas) == 3
        and all(tem_texto(programa) for programa in programas)
    )
    todos_usam_controle = (
        tres_programas_validos
        and all(
            re.search(r"(?m)^\s*(?:se|enquanto)\b", programa)
            for programa in programas
        )
    )
    checagens = [
        ("nome e domínio preenchidos", identidade_valida),
        ("pelo menos 3 tipos", quantidade_itens("tipos") >= 3),
        ("pelo menos 2 comandos do domínio", quantidade_itens("comandos_dominio") >= 2),
        ("pelo menos 2 regras de negócio", quantidade_itens("regras_de_negocio") >= 2),
        ("exatamente 3 programas-exemplo", tres_programas_validos),
        ("cada programa usa se ou enquanto", todos_usam_controle),
    ]

    for texto, ok in checagens:
        print(("✅" if ok else "⬜"), texto)

    faltam = [texto for texto, ok in checagens if not ok]
    if faltam:
        raise ValueError("Ficha da linguagem incompleta: " + "; ".join(faltam))

    print("\n🚀 Ficha completa! O programa de exemplo pode ser carregado.")


conferir_ficha(MINHA_FICHA)

# ============================================================
# PASSO 0 — O "programa dos sonhos" da FidelidadeScript
# ============================================================

PROGRAMA_FIDELIDADE = r'''
// FidelidadeScript — campanha de cashback e pontos
decimal compra = 250.0;
decimal percentual_cashback = 10.0;
decimal cashback = compra * percentual_cashback / 100.0;
inteiro pontos = 100;

se compra >= 200.0 {
    pontos = pontos + 50;
}

mostrar "Cashback calculado:";
mostrar cashback;
creditar "cliente_42", pontos;
extrato "cliente_42";
resgatar "cupom_10", 100;
expirar "cliente_42", 25;
extrato "cliente_42";
'''

print("Programa com", PROGRAMA_FIDELIDADE.count("\n") - 1, "linhas carregado em PROGRAMA_FIDELIDADE.")

## Passo 1 — Análise léxica

Nesta etapa, o Lark reconhece os tipos, as palavras reservadas e os símbolos da FidelidadeScript. Comentários e espaços são ignorados; erros léxicos informam localização e dica.

In [ ]:
# ============================================================
# PASSO 1 — Análise léxica da FidelidadeScript
# ============================================================

import re
from dataclasses import dataclass

from lark import Lark
from lark.exceptions import UnexpectedCharacters

TERMINAIS = r'''
// ---------- palavras reservadas e tipos ----------
T_INTEIRO: "inteiro"
T_DECIMAL: "decimal"
T_TEXTO: "texto"
T_LOGICO: "logico"
MOSTRAR: "mostrar"
CREDITAR: "creditar"
RESGATAR: "resgatar"
EXPIRAR: "expirar"
EXTRATO: "extrato"
SE: "se"
SENAO: "senao"
ENQUANTO: "enquanto"
E: "e"
OU: "ou"
NAO: "nao"
VERDADEIRO: "verdadeiro"
FALSO: "falso"

// ---------- literais e identificadores ----------
NUM_DECIMAL.2: /[0-9]+\.[0-9]+/
NUM_INTEIRO: /[0-9]+/
TEXTO_LIT: /"[^"\n]*"/
NOME: /(?!(?:inteiro|decimal|texto|logico|mostrar|creditar|resgatar|expirar|extrato|se|senao|enquanto|e|ou|nao|verdadeiro|falso)\b)[a-zA-Z][a-zA-Z0-9_]*/

// ---------- operadores ----------
MAIOR_IGUAL: ">="
MENOR_IGUAL: "<="
IGUAL_IGUAL: "=="
DIFERENTE: "!="
MAIOR: ">"
MENOR: "<"
ATRIB: "="
MAIS: "+"
MENOS: "-"
VEZES: "*"
DIVIDE: "/"

// ---------- pontuação ----------
PONTO_VIRGULA: ";"
VIRGULA: ","
ABRE_CHAVE: "{"
FECHA_CHAVE: "}"
ABRE_PAR: "("
FECHA_PAR: ")"

// ---------- elementos ignorados ----------
COMENTARIO: /\/\/[^\n]*/
%import common.WS
%ignore WS
%ignore COMENTARIO
'''

PALAVRAS_RESERVADAS = {
    "inteiro": "T_INTEIRO",
    "decimal": "T_DECIMAL",
    "texto": "T_TEXTO",
    "logico": "T_LOGICO",
    "mostrar": "MOSTRAR",
    "creditar": "CREDITAR",
    "resgatar": "RESGATAR",
    "expirar": "EXPIRAR",
    "extrato": "EXTRATO",
    "se": "SE",
    "senao": "SENAO",
    "enquanto": "ENQUANTO",
    "e": "E",
    "ou": "OU",
    "nao": "NAO",
    "verdadeiro": "VERDADEIRO",
    "falso": "FALSO",
}

NOMES_TERMINAIS = [
    nome
    for nome in re.findall(r"^([A-Z_]+)(?:\.\d+)?:", TERMINAIS, re.MULTILINE)
    if nome != "COMENTARIO"
]

if len(NOMES_TERMINAIS) < 20:
    raise RuntimeError("A FidelidadeScript precisa definir pelo menos 20 tipos de token.")
if len(PALAVRAS_RESERVADAS) < 8 or not {"creditar", "resgatar", "expirar", "extrato"}.issubset(PALAVRAS_RESERVADAS):
    raise RuntimeError("Faltam palavras reservadas ou comandos do domínio no léxico.")

_lexer_lark = Lark(
    "start: token*\ntoken: " + " | ".join(NOMES_TERMINAIS) + "\n" + TERMINAIS,
    parser="lalr",
    lexer="basic",
)

@dataclass
class Diagnostico:
    fase: str
    mensagem: str
    linha: int
    coluna: int
    dica: str = ""

    def __str__(self):
        local = f"linha {self.linha}, coluna {self.coluna}"
        texto = f"[{self.fase}] {local}: {self.mensagem}"
        return texto + (f"\nDica: {self.dica}" if self.dica else "")

class ErroCompilacao(Exception):
    def __init__(self, diagnosticos):
        self.diagnosticos = diagnosticos
        super().__init__("\n".join(str(item) for item in diagnosticos))

DICAS_LEXICAS = {
    "'": 'Textos usam aspas duplas, por exemplo: "cliente_42".',
    '"': "Literal de texto sem fechamento; feche as aspas na mesma linha.",
    "&": "Para conjunção lógica, use a palavra reservada 'e'.",
    "|": "Para disjunção lógica, use a palavra reservada 'ou'.",
    "!": "Para negação, use 'nao'; para diferença, use '!='.",
    "#": "Comentários começam com //.",
    "%": "Calcule cashback com percentual / 100; '%' não é um operador da linguagem.",
}

def dica_lexica(caractere):
    if caractere in DICAS_LEXICAS:
        return DICAS_LEXICAS[caractere]
    if caractere.isalpha():
        return "Identificadores usam letras sem acento; por exemplo, escreva 'compra'."
    return "Remova o símbolo ou use um operador/pontuação definido pela FidelidadeScript."

def tokenizar(fonte):
    try:
        return list(_lexer_lark.lex(fonte))
    except UnexpectedCharacters as erro:
        caractere = erro.char
        diagnostico = Diagnostico(
            "léxico",
            f"caractere inesperado {caractere!r}",
            erro.line,
            erro.column,
            dica_lexica(caractere),
        )
        raise ErroCompilacao([diagnostico]) from None

def mostrar_tokens(fonte, limite=None):
    tokens = tokenizar(fonte)
    exibidos = tokens if limite is None else tokens[:limite]
    print(f"{'#':>3}  {'TIPO':<16} {'LEXEMA':<24} LINHA:COLUNA")
    print("-" * 62)
    for indice, token in enumerate(exibidos, 1):
        print(f"{indice:>3}  {token.type:<16} {token.value:<24} {token.line}:{token.column}")
    return tokens

print(f"Tipos de token: {len(NOMES_TERMINAIS)} | Palavras reservadas: {len(PALAVRAS_RESERVADAS)}")
print("Tabela de tokens de PROGRAMA_FIDELIDADE:")
TOKENS_PROGRAMA = mostrar_tokens(PROGRAMA_FIDELIDADE)

ERROS_LEXICOS_DEMONSTRACAO = [
    ("aspas simples", "texto cliente = 'cliente_42';"),
    ("operador lógico inválido", "se pontos > 0 & compra > 0 {}"),
    ("identificador com acento", "inteiro prêmio = 10;"),
]

for nome_caso, fonte_errada in ERROS_LEXICOS_DEMONSTRACAO:
    try:
        tokenizar(fonte_errada)
    except ErroCompilacao as erro:
        print(f"\nExemplo de erro ({nome_caso}):\n{erro}")
    else:
        raise AssertionError(f"O exemplo de erro lexical '{nome_caso}' deveria falhar.")